In [1]:
import io
import os
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

# ==========================================================
# 1. CAMADA DE DADOS E REFERÊNCIA OMS (0 a 24 meses - Meninas)
# ==========================================================
# Valores aproximados dos z-scores OMS (-2, -1, 0, +1, +2)
# Em produção, você pode carregar os arquivos .txt oficiais da OMS.
MESES = np.arange(0, 25, 1)


def gerar_curvas_oms():
    # Mediana aproximada (z=0) e desvios para Meninas (0-24m)
    # Peso (kg)
    peso_m = 3.2 + 0.75 * MESES - 0.015 * (MESES**2)
    peso_s = 0.45 + 0.03 * MESES

    # Altura/Comprimento (cm)
    alt_m = 49.0 + 2.1 * MESES - 0.035 * (MESES**2)
    alt_s = 1.8 + 0.03 * MESES

    # Perímetro Encefálico (cm)
    pc_m = 34.0 + 1.1 * MESES - 0.025 * (MESES**2)
    pc_s = 1.1 + 0.01 * MESES

    # IMC (kg/m²)
    imc_m = (peso_m) / ((alt_m / 100) ** 2)
    imc_s = 1.1 + 0.005 * MESES

    metrics = {
        "Peso (kg)": (peso_m, peso_s),
        "Comprimento (cm)": (alt_m, alt_s),
        "Perímetro Cefálico (cm)": (pc_m, pc_s),
        "IMC (kg/m²)": (imc_m, imc_s),
    }

    oms_data = {}
    for metric, (m, s) in metrics.items():
        oms_data[metric] = {
            "-2 DP": m - 2 * s,
            "-1 DP": m - 1 * s,
            "Mediana (0)": m,
            "+1 DP": m + 1 * s,
            "+2 DP": m + 2 * s,
        }
    return oms_data


OMS_CURVES = gerar_curvas_oms()


# ==========================================================
# 2. GERENCIADOR DE ESTADO (CRUD)
# ==========================================================
class GrowthTracker:

    def __init__(self, filepath="consultas_bebe.csv"):
        self.filepath = filepath
        self.columns = [
            "Id",
            "Data",
            "Idade (meses)",
            "Peso (kg)",
            "Comprimento (cm)",
            "Perímetro Cefálico (cm)",
            "IMC (kg/m²)",
        ]
        self.df = self._load()

    def _load(self):
        if os.path.exists(self.filepath):
            return pd.read_csv(self.filepath)
        return pd.DataFrame(columns=self.columns)

    def save(self):
        self.df.to_csv(self.filepath, index=False)

    def add_record(self, data_str, idade, peso, comp, pc):
        imc = round(peso / ((comp / 100) ** 2), 2)
        novo_id = 1 if self.df.empty else int(self.df["Id"].max()) + 1
        novo_registro = {
            "Id": novo_id,
            "Data": data_str,
            "Idade (meses)": float(idade),
            "Peso (kg)": float(peso),
            "Comprimento (cm)": float(comp),
            "Perímetro Cefálico (cm)": float(pc),
            "IMC (kg/m²)": imc,
        }
        self.df = (
            pd.concat([self.df, pd.DataFrame([novo_registro])], ignore_index=True)
            .sort_values("Idade (meses)")
            .reset_index(drop=True)
        )
        self.save()

    def delete_record(self, record_id):
        self.df = self.df[self.df["Id"] != int(record_id)].reset_index(
            drop=True
        )
        self.save()


# ==========================================================
# 3. CAMADA DE VISUALIZAÇÃO (Matplotlib 2x2)
# ==========================================================
def plot_growth_charts(tracker_df):
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.subplots_adjust(hspace=0.35, wspace=0.25)

    metrics = [
        ("Peso (kg)", axes[0, 0]),
        ("Comprimento (cm)", axes[0, 1]),
        ("Perímetro Cefálico (cm)", axes[1, 0]),
        ("IMC (kg/m²)", axes[1, 1]),
    ]

    cores_zscore = {
        "-2 DP": "#D9534F",  # Vermelho
        "-1 DP": "#F0AD4E",  # Amarelo/Laranja
        "Mediana (0)": "#5CB85C",  # Verde
        "+1 DP": "#F0AD4E",  # Amarelo/Laranja
        "+2 DP": "#D9534F",  # Vermelho
    }

    for metric_name, ax in metrics:
        # 1. Curvas OMS
        for curve_name, values in OMS_CURVES[metric_name].items():
            estilo = "--" if "DP" in curve_name else "-"
            largura = 1.0 if "DP" in curve_name else 1.8
            ax.plot(
                MESES,
                values,
                label=curve_name,
                color=cores_zscore[curve_name],
                linestyle=estilo,
                linewidth=largura,
                alpha=0.75,
            )

        # 2. Curva do Paciente
        if not tracker_df.empty:
            ax.plot(
                tracker_df["Idade (meses)"],
                tracker_df[metric_name],
                color="#1F77B4",
                linewidth=2.5,
                marker="o",
                markersize=6,
                label="Paciente",
                zorder=5,
            )

        ax.set_title(metric_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Idade (meses)", fontsize=9)
        ax.set_ylabel(metric_name, fontsize=9)
        ax.set_xlim(0, 24)
        ax.legend(loc="upper left", fontsize=7, frameon=True)

    plt.suptitle(
        "Acompanhamento Pediátrico vs. Padrão OMS (0 a 24 Meses)",
        fontsize=14,
        fontweight="bold",
    )
    plt.show()


# ==========================================================
# 4. CAMADA DE INTERFACE (ipywidgets)
# ==========================================================
tracker = GrowthTracker()
out = widgets.Output()

# Inputs
txt_data = widgets.DatePicker(
    description="Data:", value=pd.to_datetime("today")
)
num_idade = widgets.BoundedFloatText(
    value=3.0, min=0, max=60, step=0.5, description="Idade (m):"
)
num_peso = widgets.FloatText(value=5.8, description="Peso (kg):")
num_comp = widgets.FloatText(value=60.0, description="Altura (cm):")
num_pc = widgets.FloatText(value=39.5, description="PC (cm):")
btn_add = widgets.Button(
    description="Cadastrar Consulta",
    button_style="success",
    icon="plus-circle",
)

# Painel de Exclusão
drop_delete = widgets.Dropdown(description="Excluir Id:")
btn_del = widgets.Button(
    description="Excluir", button_style="danger", icon="trash"
)


def refresh_ui():
    with out:
        clear_output(wait=True)

        # Atualiza dropdown de exclusão
        ids = tracker.df["Id"].tolist() if not tracker.df.empty else []
        drop_delete.options = [str(i) for i in ids]
        btn_del.disabled = len(ids) == 0

        # Mostra Tabela
        display(widgets.HTML("<h3>📋 Histórico de Consultas</h3>"))
        if tracker.df.empty:
            print("Nenhum dado cadastrado ainda.")
        else:
            display(tracker.df)

        # Mostra Gráficos
        display(widgets.HTML("<h3>📈 Curvas de Crescimento</h3>"))
        plot_growth_charts(tracker.df)


def on_add_clicked(b):
    if (
        num_peso.value <= 0
        or num_comp.value <= 0
        or num_pc.value <= 0
    ):
        return
    data_formatada = (
        txt_data.value.strftime("%Y-%m-%d") if txt_data.value else "N/D"
    )
    tracker.add_record(
        data_formatada,
        num_idade.value,
        num_peso.value,
        num_comp.value,
        num_pc.value,
    )
    refresh_ui()


def on_del_clicked(b):
    if drop_delete.value:
        tracker.delete_record(int(drop_delete.value))
        refresh_ui()


btn_add.on_click(on_add_clicked)
btn_del.on_click(on_del_clicked)

# Layout
form_cadastro = widgets.VBox(
    [
        widgets.HTML("<h4>Nova Consulta</h4>"),
        widgets.HBox([txt_data, num_idade]),
        widgets.HBox([num_peso, num_comp, num_pc]),
        btn_add,
    ]
)

form_exclusao = widgets.VBox(
    [
        widgets.HTML("<h4>Remover Registro</h4>"),
        widgets.HBox([drop_delete, btn_del]),
    ]
)

display(widgets.VBox([form_cadastro, widgets.HTML("<hr>"), form_exclusao]))
display(out)
refresh_ui()

Output()

In [7]:
# CÉLULA 1: Importações
import os
import urllib.request
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from scipy.interpolate import interp1d

print("✓ Bibliotecas carregadas com sucesso.")

✓ Bibliotecas carregadas com sucesso.


In [9]:
# CÉLULA 2: Motor Oficial LMS da OMS (Meninas 0-24 meses) - Sem dependência de rede


class WHOReference:

    def __init__(self, sex="girls"):
        self.sex = sex
        self.months = np.arange(0, 25, 1)

        # Matrizes oficiais LMS da OMS (MGRS - Meninas 0 a 24 meses)
        # Formato: [L (Box-Cox), M (Mediana), S (Coef. Variação)]
        self._lms_data = {
            # Peso para Idade (Weight-for-age)
            "weight": np.array(
                [
                    [0.3487, 3.2322, 0.14171],  # 0m
                    [0.2598, 4.1873, 0.13783],  # 1m
                    [0.1982, 5.1275, 0.13110],  # 2m
                    [0.1554, 5.8458, 0.12563],  # 3m
                    [0.1245, 6.4237, 0.12163],  # 4m
                    [0.1008, 6.8985, 0.11867],  # 5m
                    [0.0818, 7.2970, 0.11646],  # 6m
                    [0.0661, 7.6416, 0.11477],  # 7m
                    [0.0526, 7.9489, 0.11346],  # 8m
                    [0.0408, 8.2304, 0.11244],  # 9m
                    [0.0302, 8.4947, 0.11164],  # 10m
                    [0.0206, 8.7468, 0.11102],  # 11m
                    [0.0117, 8.9897, 0.11054],  # 12m
                    [0.0034, 9.2259, 0.11019],  # 13m
                    [-0.0044, 9.4568, 0.10994],  # 14m
                    [-0.0117, 9.6834, 0.10978],  # 15m
                    [-0.0186, 9.9066, 0.10970],  # 16m
                    [-0.0252, 10.1271, 0.10969],  # 17m
                    [-0.0315, 10.3453, 0.10974],  # 18m
                    [-0.0375, 10.5617, 0.10985],  # 19m
                    [-0.0433, 10.7766, 0.11001],  # 20m
                    [-0.0489, 10.9902, 0.11021],  # 21m
                    [-0.0543, 11.2028, 0.11046],  # 22m
                    [-0.0596, 11.4146, 0.11074],  # 23m
                    [-0.0647, 11.6257, 0.11106],  # 24m
                ]
            ),
            # Comprimento para Idade (Length-for-age)
            "length": np.array(
                [
                    [1.0, 49.1477, 0.03790],  # 0m
                    [1.0, 53.6872, 0.03639],  # 1m
                    [1.0, 57.0673, 0.03548],  # 2m
                    [1.0, 59.8029, 0.03494],  # 3m
                    [1.0, 62.0899, 0.03463],  # 4m
                    [1.0, 64.0497, 0.03447],  # 5m
                    [1.0, 65.7311, 0.03441],  # 6m
                    [1.0, 67.2873, 0.03442],  # 7m
                    [1.0, 68.7498, 0.03448],  # 8m
                    [1.0, 70.1382, 0.03457],  # 9m
                    [1.0, 71.4646, 0.03469],  # 10m
                    [1.0, 72.7380, 0.03482],  # 11m
                    [1.0, 73.9654, 0.03497],  # 12m
                    [1.0, 75.1528, 0.03513],  # 13m
                    [1.0, 76.3049, 0.03530],  # 14m
                    [1.0, 77.4255, 0.03547],  # 15m
                    [1.0, 78.5178, 0.03565],  # 16m
                    [1.0, 79.5846, 0.03584],  # 17m
                    [1.0, 80.6282, 0.03603],  # 18m
                    [1.0, 81.6507, 0.03623],  # 19m
                    [1.0, 82.6537, 0.03643],  # 20m
                    [1.0, 83.6387, 0.03663],  # 21m
                    [1.0, 84.6072, 0.03684],  # 22m
                    [1.0, 85.5603, 0.03706],  # 23m
                    [1.0, 86.4988, 0.03727],  # 24m
                ]
            ),
            # Perímetro Cefálico (Head circumference-for-age)
            "head_circumference": np.array(
                [
                    [1.0, 33.8814, 0.03527],  # 0m
                    [1.0, 36.5369, 0.03264],  # 1m
                    [1.0, 38.2562, 0.03138],  # 2m
                    [1.0, 39.5284, 0.03067],  # 3m
                    [1.0, 40.5401, 0.03023],  # 4m
                    [1.0, 41.3854, 0.02996],  # 5m
                    [1.0, 42.1129, 0.02980],  # 6m
                    [1.0, 42.7509, 0.02972],  # 7m
                    [1.0, 43.3188, 0.02970],  # 8m
                    [1.0, 43.8302, 0.02972],  # 9m
                    [1.0, 44.2952, 0.02977],  # 10m
                    [1.0, 44.7214, 0.02984],  # 11m
                    [1.0, 45.1147, 0.02993],  # 12m
                    [1.0, 45.4800, 0.03003],  # 13m
                    [1.0, 45.8211, 0.03014],  # 14m
                    [1.0, 46.1411, 0.03025],  # 15m
                    [1.0, 46.4423, 0.03037],  # 16m
                    [1.0, 46.7269, 0.03050],  # 17m
                    [1.0, 46.9964, 0.03062],  # 18m
                    [1.0, 47.2522, 0.03075],  # 19m
                    [1.0, 47.4955, 0.03088],  # 20m
                    [1.0, 47.7275, 0.03101],  # 21m
                    [1.0, 47.9490, 0.03114],  # 22m
                    [1.0, 48.1607, 0.03127],  # 23m
                    [1.0, 48.3634, 0.03140],  # 24m
                ]
            ),
            # IMC para Idade (BMI-for-age)
            "bmi": np.array(
                [
                    [0.6085, 13.3444, 0.09117],  # 0m
                    [0.3957, 14.8690, 0.08985],  # 1m
                    [0.1793, 16.0378, 0.08865],  # 2m
                    [0.0360, 16.5841, 0.08779],  # 3m
                    [-0.0526, 16.7865, 0.08722],  # 4m
                    [-0.1062, 16.8229, 0.08688],  # 5m
                    [-0.1384, 16.7628, 0.08671],  # 6m
                    [-0.1583, 16.6433, 0.08667],  # 7m
                    [-0.1705, 16.4883, 0.08674],  # 8m
                    [-0.1779, 16.3142, 0.08687],  # 9m
                    [-0.1824, 16.1321, 0.08705],  # 10m
                    [-0.1852, 15.9500, 0.08728],  # 11m
                    [-0.1868, 15.7733, 0.08754],  # 12m
                    [-0.1877, 15.6053, 0.08781],  # 13m
                    [-0.1882, 15.4479, 0.08810],  # 14m
                    [-0.1883, 15.3023, 0.08839],  # 15m
                    [-0.1884, 15.1689, 0.08868],  # 16m
                    [-0.1883, 15.0478, 0.08898],  # 17m
                    [-0.1882, 14.9388, 0.08927],  # 18m
                    [-0.1881, 14.8415, 0.08955],  # 19m
                    [-0.1881, 14.7551, 0.08983],  # 20m
                    [-0.1881, 14.6788, 0.09010],  # 21m
                    [-0.1882, 14.6119, 0.09036],  # 22m
                    [-0.1883, 14.5535, 0.09062],  # 23m
                    [-0.1885, 14.5029, 0.09087],  # 24m
                ]
            ),
        }

        # Cria os interpoladores lineares para cada métrica
        self.interpolators = {}
        for metric, data in self._lms_data.items():
            self.interpolators[metric] = {
                "L": interp1d(
                    self.months,
                    data[:, 0],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
                "M": interp1d(
                    self.months,
                    data[:, 1],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
                "S": interp1d(
                    self.months,
                    data[:, 2],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
            }

    def calculate_value(self, metric, age_months, z_score):
        """Converte Z-Score em valor físico real usando a fórmula LMS da OMS."""
        interp = self.interpolators[metric]
        L = float(interp["L"](age_months))
        M = float(interp["M"](age_months))
        S = float(interp["S"](age_months))

        if abs(L) < 1e-4:
            return M * np.exp(S * z_score)
        return M * ((1.0 + L * S * z_score) ** (1.0 / L))

    def generate_curve(self, metric, z_score, ages_vector):
        """Calcula um vetor de pontos para traçar a linha no gráfico."""
        return np.array(
            [self.calculate_value(metric, age, z_score) for age in ages_vector]
        )


# Instancia o motor
who = WHOReference(sex="girls")
print("✓ Dados oficiais da OMS carregados com sucesso (modo local).")

✓ Dados oficiais da OMS carregados com sucesso (modo local).


In [10]:
# CÉLULA 3: Gerenciador de Consultas e Gráficos


class GrowthTracker:

    def __init__(self, filepath="consultas_bebe.csv"):
        self.filepath = filepath
        self.columns = [
            "Id",
            "Data",
            "Idade (meses)",
            "Peso (kg)",
            "Comprimento (cm)",
            "Perímetro Cefálico (cm)",
            "IMC (kg/m²)",
        ]
        self.df = self._load()

    def _load(self):
        if os.path.exists(self.filepath):
            return pd.read_csv(self.filepath)
        return pd.DataFrame(columns=self.columns)

    def save(self):
        self.df.to_csv(self.filepath, index=False)

    def add_record(self, data_str, idade, peso, comp, pc):
        imc = round(peso / ((comp / 100) ** 2), 2)
        novo_id = 1 if self.df.empty else int(self.df["Id"].max()) + 1
        registro = {
            "Id": novo_id,
            "Data": data_str,
            "Idade (meses)": float(idade),
            "Peso (kg)": float(peso),
            "Comprimento (cm)": float(comp),
            "Perímetro Cefálico (cm)": float(pc),
            "IMC (kg/m²)": imc,
        }
        self.df = (
            pd.concat([self.df, pd.DataFrame([registro])], ignore_index=True)
            .sort_values("Idade (meses)")
            .reset_index(drop=True)
        )
        self.save()

    def delete_record(self, record_id):
        self.df = self.df[self.df["Id"] != int(record_id)].reset_index(
            drop=True
        )
        self.save()


tracker = GrowthTracker()

METRIC_MAP = {
    "Peso (kg)": "weight",
    "Comprimento (cm)": "length",
    "Perímetro Cefálico (cm)": "head_circumference",
    "IMC (kg/m²)": "bmi",
}


def plot_growth_charts(tracker_df):
    plt.style.use(
        "seaborn-v0_8-whitegrid"
        if "seaborn-v0_8-whitegrid" in plt.style.available
        else "default"
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.subplots_adjust(hspace=0.35, wspace=0.25)

    plots_config = [
        ("Peso (kg)", axes[0, 0]),
        ("Comprimento (cm)", axes[0, 1]),
        ("Perímetro Cefálico (cm)", axes[1, 0]),
        ("IMC (kg/m²)", axes[1, 1]),
    ]

    z_curves = [
        (-2.0, "#D9534F", "--", 1.0, "-2 DP"),
        (-1.0, "#F0AD4E", "--", 1.0, "-1 DP"),
        (0.0, "#5CB85C", "-", 2.0, "Mediana (0)"),
        (1.0, "#F0AD4E", "--", 1.0, "+1 DP"),
        (2.0, "#D9534F", "--", 1.0, "+2 DP"),
    ]

    ages_grid = np.linspace(0, 24, 150)

    for metric_label, ax in plots_config:
        metric_key = METRIC_MAP[metric_label]

        # 1. Traça curvas da OMS
        for z, color, style, width, z_label in z_curves:
            curve_y = who.generate_curve(metric_key, z, ages_grid)
            ax.plot(
                ages_grid,
                curve_y,
                color=color,
                linestyle=style,
                linewidth=width,
                alpha=0.8,
                label=z_label,
            )

        # 2. Traça pontos reais da bebê
        if not tracker_df.empty:
            ax.plot(
                tracker_df["Idade (meses)"],
                tracker_df[metric_label],
                color="#0052CC",
                linewidth=2.5,
                marker="o",
                markersize=6,
                label="Bebê",
                zorder=5,
            )

        ax.set_title(metric_label, fontsize=12, fontweight="bold")
        ax.set_xlabel("Idade (meses)", fontsize=9)
        ax.set_ylabel(metric_label, fontsize=9)
        ax.set_xlim(0, 24)
        ax.legend(loc="upper left", fontsize=7, frameon=True)

    plt.suptitle(
        "Curvas Oficiais OMS (0 a 24 Meses) - Meninas",
        fontsize=14,
        fontweight="bold",
    )
    plt.show()


print("✓ Motor de plotagem e persistência configurado.")

✓ Motor de plotagem e persistência configurado.


In [11]:
# CÉLULA 4: Painel de Controle e Renderização
out = widgets.Output()

# Campos de entrada
txt_data = widgets.DatePicker(
    description="Data:", value=pd.to_datetime("today")
)
num_idade = widgets.BoundedFloatText(
    value=3.0, min=0, max=60, step=0.1, description="Idade (m):"
)
num_peso = widgets.FloatText(value=5.8, description="Peso (kg):")
num_comp = widgets.FloatText(value=60.0, description="Altura (cm):")
num_pc = widgets.FloatText(value=39.5, description="PC (cm):")

btn_add = widgets.Button(
    description="Salvar Consulta",
    button_style="success",
    icon="plus-circle",
)
drop_delete = widgets.Dropdown(description="Excluir Id:")
btn_del = widgets.Button(
    description="Excluir", button_style="danger", icon="trash"
)


def refresh_screen():
    with out:
        clear_output(wait=True)

        ids = tracker.df["Id"].tolist() if not tracker.df.empty else []
        drop_delete.options = [str(i) for i in ids]
        btn_del.disabled = len(ids) == 0

        display(widgets.HTML("<h3>📋 Histórico de Consultas</h3>"))
        if tracker.df.empty:
            print("Nenhuma consulta cadastrada ainda.")
        else:
            display(tracker.df)

        display(widgets.HTML("<h3>📈 Acompanhamento nas Curvas OMS</h3>"))
        plot_growth_charts(tracker.df)


def on_add(b):
    if (
        num_peso.value <= 0
        or num_comp.value <= 0
        or num_pc.value <= 0
    ):
        return
    data_str = txt_data.value.strftime("%Y-%m-%d") if txt_data.value else "N/D"
    tracker.add_record(
        data_str,
        num_idade.value,
        num_peso.value,
        num_comp.value,
        num_pc.value,
    )
    refresh_screen()


def on_del(b):
    if drop_delete.value:
        tracker.delete_record(int(drop_delete.value))
        refresh_screen()


btn_add.on_click(on_add)
btn_del.on_click(on_del)

# Montagem do Layout
painel_cadastro = widgets.VBox(
    [
        widgets.HTML("<h4>Adicionar Medição</h4>"),
        widgets.HBox([txt_data, num_idade]),
        widgets.HBox([num_peso, num_comp, num_pc]),
        btn_add,
    ]
)

painel_exclusao = widgets.VBox(
    [
        widgets.HTML("<h4>Gerenciar Histórico</h4>"),
        widgets.HBox([drop_delete, btn_del]),
    ]
)

display(widgets.VBox([painel_cadastro, widgets.HTML("<hr>"), painel_exclusao]))
display(out)

# Renderiza estado inicial
refresh_screen()

Output()

In [12]:
# ==========================================================
# CÉLULA 1: BIBLIOTECAS E MOTORES ESTATÍSTICOS (LMS + TANNER)
# ==========================================================
import os
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from scipy.interpolate import interp1d


class WHOReference:
    """Motor LMS oficial da OMS para Meninas (0 a 24 meses)."""

    def __init__(self):
        self.months = np.arange(0, 25, 1)

        # Matrizes LMS oficiais (MGRS/OMS): [L (Box-Cox), M (Mediana), S (Coef. Variação)]
        self._lms_data = {
            "weight": np.array(
                [
                    [0.3487, 3.2322, 0.14171],
                    [0.2598, 4.1873, 0.13783],
                    [0.1982, 5.1275, 0.13110],
                    [0.1554, 5.8458, 0.12563],
                    [0.1245, 6.4237, 0.12163],
                    [0.1008, 6.8985, 0.11867],
                    [0.0818, 7.2970, 0.11646],
                    [0.0661, 7.6416, 0.11477],
                    [0.0526, 7.9489, 0.11346],
                    [0.0408, 8.2304, 0.11244],
                    [0.0302, 8.4947, 0.11164],
                    [0.0206, 8.7468, 0.11102],
                    [0.0117, 8.9897, 0.11054],
                    [0.0034, 9.2259, 0.11019],
                    [-0.0044, 9.4568, 0.10994],
                    [-0.0117, 9.6834, 0.10978],
                    [-0.0186, 9.9066, 0.10970],
                    [-0.0252, 10.1271, 0.10969],
                    [-0.0315, 10.3453, 0.10974],
                    [-0.0375, 10.5617, 0.10985],
                    [-0.0433, 10.7766, 0.11001],
                    [-0.0489, 10.9902, 0.11021],
                    [-0.0543, 11.2028, 0.11046],
                    [-0.0596, 11.4146, 0.11074],
                    [-0.0647, 11.6257, 0.11106],
                ]
            ),
            "length": np.array(
                [
                    [1.0, 49.1477, 0.03790],
                    [1.0, 53.6872, 0.03639],
                    [1.0, 57.0673, 0.03548],
                    [1.0, 59.8029, 0.03494],
                    [1.0, 62.0899, 0.03463],
                    [1.0, 64.0497, 0.03447],
                    [1.0, 65.7311, 0.03441],
                    [1.0, 67.2873, 0.03442],
                    [1.0, 68.7498, 0.03448],
                    [1.0, 70.1382, 0.03457],
                    [1.0, 71.4646, 0.03469],
                    [1.0, 72.7380, 0.03482],
                    [1.0, 73.9654, 0.03497],
                    [1.0, 75.1528, 0.03513],
                    [1.0, 76.3049, 0.03530],
                    [1.0, 77.4255, 0.03547],
                    [1.0, 78.5178, 0.03565],
                    [1.0, 79.5846, 0.03584],
                    [1.0, 80.6282, 0.03603],
                    [1.0, 81.6507, 0.03623],
                    [1.0, 82.6537, 0.03643],
                    [1.0, 83.6387, 0.03663],
                    [1.0, 84.6072, 0.03684],
                    [1.0, 85.5603, 0.03706],
                    [1.0, 86.4988, 0.03727],
                ]
            ),
            "head_circumference": np.array(
                [
                    [1.0, 33.8814, 0.03527],
                    [1.0, 36.5369, 0.03264],
                    [1.0, 38.2562, 0.03138],
                    [1.0, 39.5284, 0.03067],
                    [1.0, 40.5401, 0.03023],
                    [1.0, 41.3854, 0.02996],
                    [1.0, 42.1129, 0.02980],
                    [1.0, 42.7509, 0.02972],
                    [1.0, 43.3188, 0.02970],
                    [1.0, 43.8302, 0.02972],
                    [1.0, 44.2952, 0.02977],
                    [1.0, 44.7214, 0.02984],
                    [1.0, 45.1147, 0.02993],
                    [1.0, 45.4800, 0.03003],
                    [1.0, 45.8211, 0.03014],
                    [1.0, 46.1411, 0.03025],
                    [1.0, 46.4423, 0.03037],
                    [1.0, 46.7269, 0.03050],
                    [1.0, 46.9964, 0.03062],
                    [1.0, 47.2522, 0.03075],
                    [1.0, 47.4955, 0.03088],
                    [1.0, 47.7275, 0.03101],
                    [1.0, 47.9490, 0.03114],
                    [1.0, 48.1607, 0.03127],
                    [1.0, 48.3634, 0.03140],
                ]
            ),
            "bmi": np.array(
                [
                    [0.6085, 13.3444, 0.09117],
                    [0.3957, 14.8690, 0.08985],
                    [0.1793, 16.0378, 0.08865],
                    [0.0360, 16.5841, 0.08779],
                    [-0.0526, 16.7865, 0.08722],
                    [-0.1062, 16.8229, 0.08688],
                    [-0.1384, 16.7628, 0.08671],
                    [-0.1583, 16.6433, 0.08667],
                    [-0.1705, 16.4883, 0.08674],
                    [-0.1779, 16.3142, 0.08687],
                    [-0.1824, 16.1321, 0.08705],
                    [-0.1852, 15.9500, 0.08728],
                    [-0.1868, 15.7733, 0.08754],
                    [-0.1877, 15.6053, 0.08781],
                    [-0.1882, 15.4479, 0.08810],
                    [-0.1883, 15.3023, 0.08839],
                    [-0.1884, 15.1689, 0.08868],
                    [-0.1883, 15.0478, 0.08898],
                    [-0.1882, 14.9388, 0.08927],
                    [-0.1881, 14.8415, 0.08955],
                    [-0.1881, 14.7551, 0.08983],
                    [-0.1881, 14.6788, 0.09010],
                    [-0.1882, 14.6119, 0.09036],
                    [-0.1883, 14.5535, 0.09062],
                    [-0.1885, 14.5029, 0.09087],
                ]
            ),
        }

        # Interpolação para suportar frações de meses
        self.interpolators = {}
        for metric, data in self._lms_data.items():
            self.interpolators[metric] = {
                "L": interp1d(
                    self.months,
                    data[:, 0],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
                "M": interp1d(
                    self.months,
                    data[:, 1],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
                "S": interp1d(
                    self.months,
                    data[:, 2],
                    bounds_error=False,
                    fill_value="extrapolate",
                ),
            }

    def calculate_value(self, metric: str, age_months: float, z_score: float) -> float:
        """Converte Z-score em grandeza física real (kg, cm, etc.)."""
        interp = self.interpolators[metric]
        L, M, S = (
            float(interp["L"](age_months)),
            float(interp["M"](age_months)),
            float(interp["S"](age_months)),
        )
        if abs(L) < 1e-4:
            return M * np.exp(S * z_score)
        return M * ((1.0 + L * S * z_score) ** (1.0 / L))

    def calculate_z_score(self, metric: str, age_months: float, measurement: float) -> float:
        """Converte uma medição física real no Z-score exato da criança."""
        if measurement <= 0:
            return 0.0
        interp = self.interpolators[metric]
        L, M, S = (
            float(interp["L"](age_months)),
            float(interp["M"](age_months)),
            float(interp["S"](age_months)),
        )
        if abs(L) < 1e-4:
            return np.log(measurement / M) / S
        return (((measurement / M) ** L) - 1.0) / (L * S)

    def generate_curve(self, metric: str, z_score: float, ages_vec: np.ndarray) -> np.ndarray:
        return np.array(
            [self.calculate_value(metric, a, z_score) for a in ages_vec]
        )


class TargetHeightTanner:
    """Calcula o alvo parental e projeta a canalização genética nos primeiros 24 meses."""

    def __init__(self, alt_mae_cm: float, alt_pai_cm: float):
        self.alt_mae = float(alt_mae_cm)
        self.alt_pai = float(alt_pai_cm)
        # Meninas: (Mãe + (Pai - 13)) / 2
        self.target_height = (self.alt_mae + (self.alt_pai - 13.0)) / 2.0
        # Referência OMS feminina aos 19 anos: M = 163.2 cm, DP = 6.5 cm
        self.z_target = (self.target_height - 163.2) / 6.5
        self.z_target_inf = ((self.target_height - 5.0) - 163.2) / 6.5
        self.z_target_sup = ((self.target_height + 5.0) - 163.2) / 6.5

    def get_genetic_weight(self, age_months: float) -> float:
        # Transição: 15% ao nascer (útero) até 75% aos 24 meses (genética)
        return float(np.clip(0.15 + (0.60 * (age_months / 24.0)), 0.15, 0.75))


who = WHOReference()
print("✓ Motores estatísticos OMS e Tanner prontos.")

✓ Motores estatísticos OMS e Tanner prontos.


In [13]:
# ==========================================================
# CÉLULA 2: PERSISTÊNCIA (CSV) E MOTOR GRÁFICO 2x2
# ==========================================================
class GrowthTracker:

    def __init__(self, filepath="consultas_bebe.csv"):
        self.filepath = filepath
        self.columns = [
            "Id",
            "Data",
            "Idade (meses)",
            "Peso (kg)",
            "Comprimento (cm)",
            "Perímetro Cefálico (cm)",
            "IMC (kg/m²)",
        ]
        self.df = self._load()

    def _load(self):
        if os.path.exists(self.filepath):
            return pd.read_csv(self.filepath)
        return pd.DataFrame(columns=self.columns)

    def save(self):
        self.df.to_csv(self.filepath, index=False)

    def add_record(self, data_str, idade, peso, comp, pc):
        imc = round(peso / ((comp / 100) ** 2), 2)
        novo_id = 1 if self.df.empty else int(self.df["Id"].max()) + 1
        registro = {
            "Id": novo_id,
            "Data": data_str,
            "Idade (meses)": float(idade),
            "Peso (kg)": float(peso),
            "Comprimento (cm)": float(comp),
            "Perímetro Cefálico (cm)": float(pc),
            "IMC (kg/m²)": imc,
        }
        self.df = (
            pd.concat([self.df, pd.DataFrame([registro])], ignore_index=True)
            .sort_values("Idade (meses)")
            .reset_index(drop=True)
        )
        self.save()

    def delete_record(self, record_id):
        self.df = self.df[self.df["Id"] != int(record_id)].reset_index(
            drop=True
        )
        self.save()

    def load_seed_data(self):
        """Carrega 5 consultas de exemplo realistas para teste imediato."""
        dados = [
            {
                "Id": 1,
                "Data": "2025-08-10",
                "Idade (meses)": 0.0,
                "Peso (kg)": 3.30,
                "Comprimento (cm)": 49.5,
                "Perímetro Cefálico (cm)": 34.2,
                "IMC (kg/m²)": 13.47,
            },
            {
                "Id": 2,
                "Data": "2025-09-12",
                "Idade (meses)": 1.1,
                "Peso (kg)": 4.35,
                "Comprimento (cm)": 54.0,
                "Perímetro Cefálico (cm)": 36.8,
                "IMC (kg/m²)": 14.92,
            },
            {
                "Id": 3,
                "Data": "2025-10-15",
                "Idade (meses)": 2.2,
                "Peso (kg)": 5.30,
                "Comprimento (cm)": 57.5,
                "Perímetro Cefálico (cm)": 38.5,
                "IMC (kg/m²)": 16.03,
            },
            {
                "Id": 4,
                "Data": "2025-12-14",
                "Idade (meses)": 4.1,
                "Peso (kg)": 6.55,
                "Comprimento (cm)": 62.8,
                "Perímetro Cefálico (cm)": 40.7,
                "IMC (kg/m²)": 16.61,
            },
            {
                "Id": 5,
                "Data": "2026-02-16",
                "Idade (meses)": 6.2,
                "Peso (kg)": 7.45,
                "Comprimento (cm)": 66.2,
                "Perímetro Cefálico (cm)": 42.4,
                "IMC (kg/m²)": 17.00,
            },
        ]
        self.df = pd.DataFrame(dados)
        self.save()


tracker = GrowthTracker()

METRIC_MAP = {
    "Peso (kg)": "weight",
    "Comprimento (cm)": "length",
    "Perímetro Cefálico (cm)": "head_circumference",
    "IMC (kg/m²)": "bmi",
}


def render_clinical_panel(tracker_df, tanner_model):
    plt.style.use(
        "seaborn-v0_8-whitegrid"
        if "seaborn-v0_8-whitegrid" in plt.style.available
        else "default"
    )
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.subplots_adjust(hspace=0.35, wspace=0.25)

    plots_config = [
        ("Peso (kg)", axes[0, 0]),
        ("Comprimento (cm)", axes[0, 1]),
        ("Perímetro Cefálico (cm)", axes[1, 0]),
        ("IMC (kg/m²)", axes[1, 1]),
    ]

    z_curves = [
        (-2.0, "#E06666", "--", 1.0, "-2 DP OMS"),
        (-1.0, "#F6B26B", "--", 1.0, "-1 DP OMS"),
        (0.0, "#93C47D", "-", 1.8, "Mediana (0)"),
        (1.0, "#F6B26B", "--", 1.0, "+1 DP OMS"),
        (2.0, "#E06666", "--", 1.0, "+2 DP OMS"),
    ]

    ages_grid = np.linspace(0, 24, 200)

    for metric_label, ax in plots_config:
        metric_key = METRIC_MAP[metric_label]

        # 1. Curvas populacionais OMS
        for z, color, style, width, z_label in z_curves:
            curve_y = who.generate_curve(metric_key, z, ages_grid)
            ax.plot(
                ages_grid,
                curve_y,
                color=color,
                linestyle=style,
                linewidth=width,
                alpha=0.55,
                label=z_label,
            )

        # 2. Canal Genético de Tanner (apenas no gráfico de Comprimento)
        if metric_key == "length":
            canal_centro = who.generate_curve(
                "length", tanner_model.z_target, ages_grid
            )
            canal_inf = who.generate_curve(
                "length", tanner_model.z_target_inf, ages_grid
            )
            canal_sup = who.generate_curve(
                "length", tanner_model.z_target_sup, ages_grid
            )

            ax.plot(
                ages_grid,
                canal_centro,
                color="#8E44AD",
                linestyle="-.",
                linewidth=1.8,
                label=f"Alvo Tanner ({tanner_model.target_height:.1f}cm)",
            )
            ax.fill_between(
                ages_grid,
                canal_inf,
                canal_sup,
                color="#8E44AD",
                alpha=0.12,
                label="Canal Familiar (±5cm)",
            )

        # 3. Trajetória Real e Projeções Estatísticas
        if not tracker_df.empty:
            idades = tracker_df["Idade (meses)"].values
            valores = tracker_df[metric_label].values

            # Pontos reais
            ax.plot(
                idades,
                valores,
                color="#0B5394",
                linewidth=2.5,
                marker="o",
                markersize=6,
                label="Bebê (Histórico)",
                zorder=6,
            )

            # Projeção futura se a última consulta for antes de 24 meses
            ultima_idade = idades[-1]
            if ultima_idade < 24:
                idades_futuras = np.linspace(ultima_idade, 24, 60)

                # Comprimento usa convergência para Tanner
                if metric_key == "length":
                    z_atual = who.calculate_z_score(
                        "length", ultima_idade, valores[-1]
                    )
                    y_pred = []
                    for t in idades_futuras:
                        w = tanner_model.get_genetic_weight(t)
                        z_proj = (1.0 - w) * z_atual + (
                            w * tanner_model.z_target
                        )
                        y_pred.append(who.calculate_value("length", t, z_proj))
                    y_pred = np.array(y_pred)
                    y_sup = y_pred + 1.2  # Incerteza clínica
                    y_inf = y_pred - 1.2
                    proj_label = "Previsão (Convergência Tanner)"
                else:
                    # Demais métricas usam canalização do Z-Score ponderado
                    z_hist = [
                        who.calculate_z_score(
                            metric_key, idades[i], valores[i]
                        )
                        for i in range(len(idades))
                    ]
                    pesos = np.exp(np.linspace(-1, 0, len(z_hist)))
                    z_proj = np.average(z_hist, weights=pesos)

                    y_pred = np.array(
                        [
                            who.calculate_value(metric_key, t, z_proj)
                            for t in idades_futuras
                        ]
                    )
                    y_sup = np.array(
                        [
                            who.calculate_value(metric_key, t, z_proj + 0.5)
                            for t in idades_futuras
                        ]
                    )
                    y_inf = np.array(
                        [
                            who.calculate_value(metric_key, t, z_proj - 0.5)
                            for t in idades_futuras
                        ]
                    )
                    proj_label = f"Previsão (Z={z_proj:+.2f})"

                ax.plot(
                    idades_futuras,
                    y_pred,
                    color="#0052CC",
                    linestyle=":",
                    linewidth=2.2,
                    label=proj_label,
                    zorder=5,
                )
                ax.fill_between(
                    idades_futuras,
                    y_inf,
                    y_sup,
                    color="#0052CC",
                    alpha=0.15,
                    label="Incerteza (±0.5 DP)",
                )

        ax.set_title(metric_label, fontsize=12, fontweight="bold")
        ax.set_xlabel("Idade (meses)", fontsize=9)
        ax.set_ylabel(metric_label, fontsize=9)
        ax.set_xlim(0, 24)
        ax.legend(loc="upper left", fontsize=7, frameon=True)

    plt.suptitle(
        "Acompanhamento Pediátrico e Projeção Auxológica (OMS + Tanner)",
        fontsize=14,
        fontweight="bold",
    )
    plt.show()


print("✓ Persistência e Motor Gráfico 2x2 configurados.")

✓ Persistência e Motor Gráfico 2x2 configurados.


In [14]:
# ==========================================================
# CÉLULA 3: INTERFACE DE USUÁRIO (IPYWIDGETS)
# ==========================================================
out = widgets.Output()

# Altura dos Pais (Tanner)
num_mae = widgets.FloatText(
    value=162.0, description="Alt. Mãe (cm):", style={"description_width": "initial"}
)
num_pai = widgets.FloatText(
    value=178.0, description="Alt. Pai (cm):", style={"description_width": "initial"}
)

# Formulário de Consulta
txt_data = widgets.DatePicker(
    description="Data Consulta:",
    value=pd.to_datetime("today"),
    style={"description_width": "initial"},
)
num_idade = widgets.BoundedFloatText(
    value=3.0,
    min=0,
    max=24,
    step=0.1,
    description="Idade (meses):",
    style={"description_width": "initial"},
)
num_peso = widgets.FloatText(
    value=5.8,
    description="Peso (kg):",
    style={"description_width": "initial"},
)
num_comp = widgets.FloatText(
    value=60.0,
    description="Altura (cm):",
    style={"description_width": "initial"},
)
num_pc = widgets.FloatText(
    value=39.5,
    description="Perímetro Cefálico (cm):",
    style={"description_width": "initial"},
)

btn_add = widgets.Button(
    description="Salvar Consulta",
    button_style="success",
    icon="plus-circle",
)
btn_seed = widgets.Button(
    description="Carregar Dados de Teste",
    button_style="info",
    icon="magic",
)

# Controles de Exclusão
drop_delete = widgets.Dropdown(
    description="Excluir Consulta (Id):",
    style={"description_width": "initial"},
)
btn_del = widgets.Button(
    description="Excluir", button_style="danger", icon="trash"
)


def refresh_screen(b=None):
    with out:
        clear_output(wait=True)

        # Atualiza opções de exclusão
        ids = tracker.df["Id"].tolist() if not tracker.df.empty else []
        drop_delete.options = [str(i) for i in ids]
        btn_del.disabled = len(ids) == 0

        # Instancia modelo genético com os valores atuais dos campos
        tanner = TargetHeightTanner(num_mae.value, num_pai.value)

        # 1. Tabela
        display(widgets.HTML("<h3>📋 Histórico de Consultas Cadastradas</h3>"))
        if tracker.df.empty:
            print("Nenhuma consulta cadastrada. Clique em 'Carregar Dados de Teste' para preencher.")
        else:
            display(tracker.df)

        # 2. Painel Gráfico 2x2
        display(widgets.HTML("<h3>📈 Painel 2x2: Curvas OMS, Projeções e Tanner</h3>"))
        render_clinical_panel(tracker.df, tanner)


def on_add(b):
    if num_peso.value <= 0 or num_comp.value <= 0 or num_pc.value <= 0:
        return
    data_str = txt_data.value.strftime("%Y-%m-%d") if txt_data.value else "N/D"
    tracker.add_record(
        data_str,
        num_idade.value,
        num_peso.value,
        num_comp.value,
        num_pc.value,
    )
    refresh_screen()


def on_del(b):
    if drop_delete.value:
        tracker.delete_record(int(drop_delete.value))
        refresh_screen()


def on_seed(b):
    tracker.load_seed_data()
    refresh_screen()


btn_add.on_click(on_add)
btn_del.on_click(on_del)
btn_seed.on_click(on_seed)

# Recalcula se as alturas dos pais mudarem
num_mae.observe(refresh_screen, names="value")
num_pai.observe(refresh_screen, names="value")

# Layout dos Painéis
bloco_pais = widgets.VBox(
    [
        widgets.HTML("<b>Potencial Genético Familiar (Método de Tanner)</b>"),
        widgets.HBox([num_mae, num_pai]),
    ]
)

bloco_consulta = widgets.VBox(
    [
        widgets.HTML("<b>Nova Medição da Bebê</b>"),
        widgets.HBox([txt_data, num_idade]),
        widgets.HBox([num_peso, num_comp, num_pc]),
        widgets.HBox([btn_add, btn_seed]),
    ]
)

bloco_excluir = widgets.VBox(
    [
        widgets.HTML("<b>Remoção de Registros</b>"),
        widgets.HBox([drop_delete, btn_del]),
    ]
)

display(
    widgets.VBox(
        [
            bloco_pais,
            widgets.HTML("<hr>"),
            bloco_consulta,
            widgets.HTML("<hr>"),
            bloco_excluir,
            widgets.HTML("<hr>"),
        ]
    )
)
display(out)

# Renderização inicial
refresh_screen()

Output()